In [6]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit

from langchain_classic.tools.retriever import create_retriever_tool

from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.utilities import SQLDatabase

from langchain_core.output_parsers import StrOutputParser, JsonOutputParser, PydanticOutputParser

from langchain_community.tools import ListDirectoryTool, ReadFileTool, WriteFileTool, CopyFileTool, DeleteFileTool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.tools import WikipediaQueryRun

from langchain_classic.chains import LLMMathChain

from langchain_core.documents import Document

from langchain_core.runnables import RunnablePassthrough, RunnableLambda, RunnableParallel

from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder

from langchain_core.tools import StructuredTool
from langchain_core.tools import Tool
from langchain_core.tools import tool

from langchain.agents import create_agent

from langchain_chroma import Chroma

from langchain_openai import ChatOpenAI

from langchain_ollama import ChatOllama

from langchain_ibm import WatsonxEmbeddings
from langchain_ibm import ChatWatsonx

from pydantic import BaseModel, Field

from dotenv import load_dotenv

import sqlite3

import os

In [2]:
#.env 내용 가죠오기
load_dotenv()

apikey = os.getenv("WATSONX_API_KEY")
project_id = os.getenv("WATSONX_PROJECT_ID")
watsonx_ai_url = os.getenv("WATSONX_URL")
hf_token = os.getenv("HF_TOKEN")
cohere_api_key = os.getenv("COHERE_API_KEY")
serper_api_key = os.getenv("SERPER_API_KEY")

# HuggingFace model

hugging_llm = ChatOpenAI(
    model="Qwen/Qwen2.5-7B-Instruct:together",
    api_key=hf_token,
    base_url="https://router.huggingface.co/v1",
    temperature= 0
)
# 유료 LLM 선언

watson_llm = ChatWatsonx(
    model_id="ibm/granite-4-h-small",
    url = f"{watsonx_ai_url}",
    api_key = f"{apikey}",
    project_id=f"{project_id}",
    params = {
    "max_tokens": 2000,
    "temperature": 0
    }
)

# 로컬 LLM 선언
qwen_llm = ChatOllama(model="qwen3.5:4b",temperature= 0)

exaone_llm = ChatOllama(model="exaone3.5:2.4b",temperature= 0)

watsonx_enbedding = WatsonxEmbeddings(
    model_id="ibm/granite-embedding-278m-multilingual",
    url = f"{watsonx_ai_url}",
    api_key = f"{apikey}",
    project_id=f"{project_id}"
)

#### RouterChain(LCEL Router: 조건부 분기 패턴)
- Router 패턴: 입력에 따라 적절한 chain or Runnable 로 분기
- 분류기가 입력 분석(LLM 기반 or 규칙 기반) -> 적절한 chain으로 Routting
- |
- ex) 질문 유형(코딩 / 수학 / 일반 / ....)에 따라 다른 전문 프롬프트를 적용
- 동작 흐름
    (1) 입력 => (2) 분류 => (3) Routting => (4) 실행 => (5) 반환

In [8]:
parsor = StrOutputParser()

#수학 체인
math_chain = ChatPromptTemplate.from_messages([
    ("system"," 당신은 수학 전문가입니다. 풀이 과정을 단계별로 설명하세요."),
    ("human","{question}")
]) | watson_llm|parsor

#코드 체인
code_chain = ChatPromptTemplate.from_messages([
    ("system"," 당신은 시니어 개발자입니다. 코드와 주석을 함께 제공하세요."),
    ("human","{question}")
]) | watson_llm|parsor
#일반 체인
general_chain = ChatPromptTemplate.from_messages([
    ("system"," 당신은 친절한 AI 어시스턴트입니다. 한국어로 답변하세요."),
    ("human","{question}")
]) | watson_llm|parsor

# 질문을 읽고 유형 반환
classify_chain = ChatPromptTemplate.from_messages([
    ("system","질문 유형을 math/code/general 중 하나로만 답하세요."),
    ("human","{question}")
]) | watson_llm|parsor

# 라우터 함수
def route(inputs:dict):
    category = classify_chain.invoke(inputs).strip().lower()
    print(f" => 분류 결과 {category}")

    if 'math' in category: return math_chain
    elif 'code' in category: return code_chain
    else: return general_chain

# router_chain = RunnableLambda(route) | RunnableLambda(lambda chain: chain)
router_chain = RunnableLambda(lambda x:route(x).invoke(x))

In [9]:
questions = [
    {'question':'피타고라스 정리 증명해줘'},
    {'question':'오늘 저녁 메뉴 추천해줘'},
    {'question':'파이썬으로 버블 정렬 구현해줘'},
]

for q in questions:
    print(f"Q: {q['question']}")
    print(f"A: {router_chain.invoke(q)[:100]}...\n")

Q: 피타고라스 정리 증명해줘
 => 분류 결과 피타고라스 정리(pythagorean theorem)는 직각삼각형에서 빗변의 길이의 제곱이 나머지 두 변의 길이의 제곱의 합과 같다는 정리입니다. 수식으로 표현하면 다음과 같습니다.

a² + b² = c²

여기서 a와 b는 직각삼각형의 두 변의 길이이고, c는 빗변의 길이입니다.

피타고라스 정리의 증명은 다양한 방법으로 이루어질 수 있으며, 그 중 하나를 소개하겠습니다.

증명:

1. 직각삼각형 abc를 그리고, 빗변 c에 대해 정사각형을 그립니다. 이 정사각형의 넓이는 c²입니다.
2. 삼각형의 두 변 a와 b에 대해 각각 정사각형을 그립니다. 이 정사각형의 넓이는 각각 a²와 b²입니다.
3. 이제 직각삼각형 abc를 a²와 b²의 정사각형 안에 넣습니다. 삼각형의 빗변 c가 정사각형의 대각선이 되도록 배치합니다.
4. a²와 b²의 정사각형 안에는 삼각형이 각각 두 개씩 들어갑니다. 따라서, 두 정사각형 안에는 총 4개의 삼각형이 들어갑니다.
5. 이제 a²와 b²의 정사각형을 결합하여 하나의 큰 정사각형을 만듭니다. 이 큰 정사각형의 한 변의 길이는 (a+b)이고, 넓이는 (a+b)²입니다.
6. 큰 정사각형 안에는 4개의 삼각형과 작은 정사각형(빗변 c가 한 변인)이 들어있습니다. 따라서, 큰 정사각형의 넓이는 4(삼각형의 넓이) + c²와 같습니다.
7. 삼각형의 넓이는 (1/2)ab이므로, 큰 정사각형의 넓이는 4((1/2)ab) + c² = 2ab + c²입니다.
8. 이제 큰 정사각형의 넓이를 두 가지 방법으로 표현한 식을 서로 비교합니다. (a+b)² = 2ab + c²
9. 식을 풀면 a² + 2ab + b² = 2ab + c²가 됩니다.
10. 양변에서 2ab를 빼면 a² + b² = c²가 됩니다.

이로써 피타고라스 정리가 증명되었습니다.
A: 피타고라스 정리는 직각삼각형에서 빗변의 길이의 제곱이 나머지 두 변의 길이의 제곱의 합과 같다는 정리입니다. 이 정리를 증명해 드리겠습니다

#### RunnableBranch(선언형 분기)

In [10]:
from langchain_core.runnables import RunnableBranch

# RunnableBranch((조건1, 체인1),(조건2, 체인2),(조건3, 체인3))

branch = RunnableBranch(
    (lambda x:'math' in  x.get('topic',''),math_chain),
    (lambda x:'code' in  x.get('topic',''),code_chain),
    (lambda x:'cooking' in  x.get('topic',''),ChatPromptTemplate.from_messages([
        ("system"," 당신은 요리 전문가입니다. 한국어로 답변하세요."),
        ("human","{question}")
    ]) | watson_llm | parsor),
    general_chain
)

print(branch.invoke({'topic':'math','question':'미분이란?'}))
print(branch.invoke({'topic':'cooking','question':'김치찌개 레시피'}))
print(branch.invoke({'topic':'other','question':'안녕하세요'}))
print(branch.invoke({'topic':'code','question':'파이썬으로 선택 정렬 구현'}))

미분은 미적분학의 한 분야로, 함수의 변화율을 나타내는 수학적 개념입니다. 미분을 통해 함수의 기울기, 즉 함수가 어떤 점에서 얼마나 빠르게 증가하거나 감소하는지를 알 수 있습니다. 이를 통해 함수의 최대값, 최소값, 곡률 등을 분석할 수 있습니다.

미분의 기본 개념은 함수의 접선의 기울기를 구하는 것입니다. 함수 f(x)가 주어졌을 때, x=a에서의 미분은 다음과 같이 정의됩니다.

f'(a) = lim(h→0) (f(a+h) - f(a))/h

여기서 f'(a)는 x=a에서의 함수 f(x)의 미분값이며, lim은 극한을 나타냅니다. 이 식은 x=a에서의 함수 f(x)의 접선의 기울기를 구하는 것과 같습니다.

미분의 표기법은 다양한데, 가장 일반적인 것은 라이프니츠 표기법과 뉴턴 표기법입니다. 라이프니츠 표기법에서는 f(x)의 미분을 df/dx 또는 f'(x)로 표기하고, 뉴턴 표기법에서는 점(dot)을 사용하여 f'(x) 또는 f''(x)와 같이 표기합니다.

미분은 다양한 응용 분야에서 사용되며, 물리학, 공학, 경제학 등에서 중요한 역할을 합니다. 예를 들어, 물리학에서는 물체의 속도와 가속도를 구하기 위해 미분을 사용하고, 경제학에서는 수요와 공급의 변화율을 분석하기 위해 미분을 사용합니다.
김치찌개는 한국 요리의 대표적인 한 그릇 요리로, 매콤하고 깊은 맛이 특징입니다. 아래는 간단한 김치찌개 레시피입니다.

재료:
- 김치 2컵
- 돼지고기 목살 200g (없으면 두부나 버섯으로 대체 가능)
- 두부 1/2모
- 대파 1대
- 양파 1/2개
- 마늘 3쪽
- 고추장 1큰술
- 고춧가루 1큰술 (선택 사항)
- 물 2컵
- 참기름 1큰술
- 소금 약간
- 찹쌀가루 또는 녹말가루 (물에 풀어 두는 것, 국물을 진하게 하기 위해)

조리 방법:

1. 돼지고기는 적당한 크기로 썰어 두세요. 두부는 네모난 조각으로 썰어 두세요.

2. 냄비에 물을 넣고 끓여줍니다. 물이 끓으면 고기를 넣고 익을 때까지 중불에서 끓입니다. 고기가 익으면 김치를 넣고 함

### SequentialChain(단계별 순차 파이프라인)
- 앞 단계 출력이 뒷 단계 입력으로 흘러 들어감
- | => SequentialChain
- .assign():중간 결과를 보존하며 뎌러 단계를 누적할 수 있음
- 각 단계의 출력 타입과 입력 타입이 일치해야함

In [ ]:
# 체인 정의
translate_chain = ChatPromptTemplate.from_messages([
    ("system",":다음 텍스트를 한국어로 번역하세요. 번역문만 출력\n{text}")
]) | watson_llm | parsor
summarize_chain = ChatPromptTemplate.from_messages([
    ("system",":다음 텍스트를 3문장으로 요약하세요.\n{text}")
]) | watson_llm | parsor
sentiment_chain = ChatPromptTemplate.from_messages([
    ("system",":다음 텍스트의 감정을 긍정/부정/중립 중 하나로만 답하세요.\n{summary}")
]) | watson_llm | parsor
report_chain = ChatPromptTemplate.from_messages([
    ("system","아래 분석 결과를 바탕으로 한 줄 최종 보고서를 작성하세요.\n"
     "원문:{text}\n번역:{translated}\n감정:{summary}\n감정:{sentiment}")
]) | watson_llm | parsor

# 원문 => 번역 => 요약 => 감정 분석 => 보고서 작성
# .assign() : 단계별 결과 누적
# translate_chain | summarize_chain | sentiment_chain | report_chain => 결과만 전달되어 loss가 심함

pipline=(RunnablePassthrough
         .assign(translated=translate_chain)
         .assign(summary=summarize_chain)
         .assign(sentiment=sentiment_chain)
         .assign(report=report_chain)
         )

result = pipline.invoke({"text":"Python is a versatile language loved by developres worldwide."})

print("번역 : ", result['translated'])
print("요약 : ", result['summary'])
print("감정 : ", result['sentiment'])
print("보고서 : ", result['report'])

번역 :  파이썬은 전 세계 개발자들이 사랑하는 다용도 언어입니다.
요약 :  파이썬은 개발자들에게 사랑받는 다용도의 언어입니다.
감정 :  긍정
보고서 :  파이썬은 개발자들에게 사랑받는 다용도의 언어입니다.


In [17]:
# 조건부 단계 삽입

detect_lang_chain = ChatPromptTemplate.from_messages([
    ("system",":다음 텍스트의 언어를 korean/english/other 중 하나로만 답하세요.\n{text}")
]) | watson_llm | parsor

def maybe_translate(inputs):
    lang =detect_lang_chain.invoke(inputs).strip().lower()

    if 'english' in lang or 'other' in lang:
        translated = translate_chain.invoke(inputs)
        return {**inputs,'text':translated, 'was_translated':True}
    return {**inputs,'was_translated':False}

smart_pipeline = (RunnableLambda(maybe_translate)
| RunnablePassthrough.assign(summary=summarize_chain)
| RunnablePassthrough.assign(sentiment=sentiment_chain))

r1=smart_pipeline.invoke({"text":"Python is Great!"})
r2=smart_pipeline.invoke({"text":"파이썬은 최고야!"})

print(r1['was_translated'],r1['summary'][:50])
print(r2['was_translated'],r2['summary'][:50])

True 파이썬은 강력하고 다재다능한 프로그래밍 언어입니다. 이를 통해 개발자는 다양한 작업을 효율
True 파이썬은 최고의 프로그래밍 언어입니다. 파이썬은 간결하고 읽기 쉬운 문법을 가지고 있어 초


### MapReduceChain(대용량 문서 처리)
- Map : 문서를  청크로 나눠 각각 처리(요약, 추출, ...) =>병렬 실행 가능
- Reduce : Map 결과들을 하나로 합쳐 최종 답변 생성

In [19]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

map_chain = ChatPromptTemplate.from_messages([
    ("system",":다음 텍스트를 2문장으로 요약하세요."),
    ("user","{chunk}")
]) | watson_llm | parsor
reduce_chain = ChatPromptTemplate.from_messages([
    ("system",":다음은 긴 문서의 섹션별 요약입니다. 전체를 5문장으로 통합 요약하세요."),
    ("user","{summaries}")
]) | watson_llm | parsor

def map_reduce_summarize(file_path):
    loader = PyPDFLoader(file_path)
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    chunks = splitter.split_documents(loader.load())
    print(f"총 청크 수 {len(chunks)}")
    #---Map : 청크별 요약
    chunk_inputs = [{'chunk':c.page_content} for c in chunks]
    summaries = map_chain.batch(chunk_inputs, config={'max_concurrency':5})
    print(f"총 청크 수 {len(summaries)} 개 요약 생성")
    #---Reduce : 요약 통합
    # 문서 결합
    combined = "\n\n".join(f"[색션 {i+1}] {s}" for i, s in enumerate(summaries))
    final = reduce_chain.invoke({'summaries':combined})
    return final

result = map_reduce_summarize("./data/직무기술서/2026 상 삼성E&A 직무기술서.pdf")
print(result)

총 청크 수 41
총 청크 수 41 개 요약 생성
삼성E&A는 다양한 분야에서 기술직(설계/조달)과 기술직(사업관리/시공관리/품질) 인턴을 모집하고 있으며, 이들은 화학/화공, 기계, 전기전자 등 다양한 분야에서 프로젝트의 전 과정을 관리하고, 품질 보증 및 안전 관리를 담당합니다. 특히, 화공플랜트와 석유화학 제품, 바이오 의약품 플랜트, 재생에너지 및 수전해 기술을 활용한 그린 수소 생산 등 다양한 사업 분야에서 혁신적인 솔루션을 제공하고 있습니다. 신입사원들은 각 분야에서 전문성을 쌓아 기술 전문가, 엔지니어링 및 프로젝트 매니저로 성장할 수 있는 기회를 제공받으며, 글로벌 수준의 역량을 갖추고 외국어(영어) 회화 능력을 갖추는 것이 중요합니다.
